# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [8]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [9]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [10]:
#drop missing values
spaceship = spaceship.dropna()
spaceship['Cabin'] = spaceship['Cabin'].astype(str).str[0]
spaceship = spaceship.drop(columns=['PassengerId', 'Name'])

In [17]:
# For non numerical values do dummies
spaceship = pd.get_dummies(spaceship, columns=['HomePlanet', 'CryoSleep','Destination','Cabin', 'VIP', 'Transported' ], drop_first=True)


**Perform Train Test Split**

In [18]:
# Perform train-test split

X = spaceship.drop(columns=['Transported_True'])
y = spaceship['Transported_True'] 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (5284, 19)
X_test shape: (1322, 19)
y_train shape: (5284,)
y_test shape: (1322,)


**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

- Bagging and Pasting

In [19]:
# Try bagging and boosting models
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import VotingClassifier
from sklearn.ensemble import StackingClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import HistGradientBoostingClassifier


In [24]:
# Bagging and Pasting
base_model = DecisionTreeClassifier(random_state=42)

# Bagging with replacement
bagging_model = BaggingClassifier(estimator=base_model, 
                                  n_estimators=100, # number of trees
                                  max_samples=0.8, # percentage of samples to draw from X to train each base estimator
                                  bootstrap=True, # with replacement
                                  random_state=42)
# Pasting without replacement
pasting_model = BaggingClassifier(estimator=base_model, 
                                  n_estimators=100, 
                                  max_samples=0.8, 
                                  bootstrap=False, # without replacement
                                  random_state=42)
# Fit and evaluate Bagging model
bagging_model.fit(X_train, y_train)
y_pred_bagging = bagging_model.predict(X_test)
print("Bagging Classifier Accuracy:", accuracy_score(y_test, y_pred_bagging))
print(classification_report(y_test, y_pred_bagging))   

# Fit and evaluate Pasting model
pasting_model.fit(X_train, y_train)
y_pred_pasting = pasting_model.predict(X_test)
print("Pasting Classifier Accuracy:", accuracy_score(y_test, y_pred_pasting))
print(classification_report(y_test, y_pred_pasting))



 


Bagging Classifier Accuracy: 0.8086232980332829
              precision    recall  f1-score   support

       False       0.80      0.81      0.81       653
        True       0.82      0.80      0.81       669

    accuracy                           0.81      1322
   macro avg       0.81      0.81      0.81      1322
weighted avg       0.81      0.81      0.81      1322

Pasting Classifier Accuracy: 0.8018154311649016
              precision    recall  f1-score   support

       False       0.79      0.82      0.80       653
        True       0.82      0.79      0.80       669

    accuracy                           0.80      1322
   macro avg       0.80      0.80      0.80      1322
weighted avg       0.80      0.80      0.80      1322



In [ ]:
# Try bagging models
models = [
    {
        "name": "Random Forest",
        "model": RandomForestClassifier(n_estimators=100)
    },
    {
        "name": "Bagging Decision Tree",
        "model": BaggingClassifier(base_estimator=DecisionTreeClassifier(), n_estimators=100)
    },
    {
        "name": "Bagging KNN",
        "model": BaggingClassifier(base_estimator=KNeighborsClassifier(), n_estimators=100)
    }
]   
results = []
for model in models:
    model_instance = model["model"]
    model_instance.fit(X_train, y_train)
    y_pred = model_instance.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred, output_dict=True)
    cm = confusion_matrix(y_test, y_pred)
    results.append({
        "model": model["name"],
        "accuracy": accuracy,
        "precision": report['1']['precision'],
        "recall": report['1']['recall'],
        "f1-score": report['1']['f1-score'],
        "confusion_matrix": cm
    })

- Random Forests

In [29]:
# Random Forest model

rf = RandomForestClassifier(
    n_estimators=100,      # number of trees
    max_samples=0.8,       # optional: fraction of training data per tree
    bootstrap=True,        # with replacement (bagging)
    random_state=42
)

# Train
rf.fit(X_train, y_train)

# Predict
y_pred_rf = rf.predict(X_test)

# Evaluate
print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))



Random Forest Accuracy: 0.8071104387291982
              precision    recall  f1-score   support

       False       0.79      0.82      0.81       653
        True       0.82      0.79      0.81       669

    accuracy                           0.81      1322
   macro avg       0.81      0.81      0.81      1322
weighted avg       0.81      0.81      0.81      1322



- Gradient Boosting

In [30]:
gb = GradientBoostingClassifier(
    n_estimators=100,      # number of boosting rounds
    learning_rate=0.1,     # step size shrinkage
    max_depth=3,           # depth of each tree
    random_state=42
)

# Train
gb.fit(X_train, y_train)

# Predict
y_pred_gb = gb.predict(X_test)

# Evaluate
print("Gradient Boosting Accuracy:", accuracy_score(y_test, y_pred_gb))
print(classification_report(y_test, y_pred_gb))

Gradient Boosting Accuracy: 0.8071104387291982
              precision    recall  f1-score   support

       False       0.84      0.76      0.79       653
        True       0.78      0.86      0.82       669

    accuracy                           0.81      1322
   macro avg       0.81      0.81      0.81      1322
weighted avg       0.81      0.81      0.81      1322



- Adaptive Boosting

In [31]:
ada = AdaBoostClassifier(
    n_estimators=100,      # number of weak learners
    learning_rate=0.1,     # contribution of each tree
    random_state=42
)

# Train
ada.fit(X_train, y_train)

# Predict
y_pred_ada = ada.predict(X_test)

# Evaluate
print("AdaBoost Accuracy:", accuracy_score(y_test, y_pred_ada))
print(classification_report(y_test, y_pred_ada))

/opt/anaconda3/lib/python3.12/site-packages/sklearn/ensemble/_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


AdaBoost Accuracy: 0.7798789712556732
              precision    recall  f1-score   support

       False       0.77      0.79      0.78       653
        True       0.79      0.77      0.78       669

    accuracy                           0.78      1322
   macro avg       0.78      0.78      0.78      1322
weighted avg       0.78      0.78      0.78      1322



Which model is the best and why?

Bagging achieved the highest accuracy (0.81), slightly outperforming Pasting, demonstrating that sampling with replacement stabilizes trees. Random Forest and Gradient Boosting gave similar accuracy (~0.81), with Gradient Boosting showing a trade-off in class-specific precision and recall. AdaBoost performed slightly worse (0.78 accuracy), likely due to sensitivity to noise in the dataset. Overall, Bagging and Random Forest provided the most balanced and stable performance across classes.